# Homework04 - Spatial Analysis with TopologicPy

This notebook reworks the original draft using the spatial-analysis patterns from:
- S03-06 Spatial Intelligence Part 1
- S03-07 Spatial Intelligence Part 2
- S03-08 Spatial Intelligence Part 3

The focus is on building a clean TopologicPy workflow for grid slicing, graph creation, centrality metrics, shortest paths, and presentation-ready interpretation.

## Workflow

- Load a floor OBJ from the Homework04 floor-plan folder.
- Show all available floor plans before selecting one.
- Build a clipped grid and slice the geometry into a shell.
- Create navigation and analysis graphs with TopologicPy.
- Compute degree, closeness, and betweenness centrality.
- Run a visibility graph analysis using isovists.
- Trace a shortest path between two representative spaces.
- Export a compact metrics table for the presentation.

In [11]:
from pathlib import Path
from time import perf_counter

import pandas as pd

from topologicpy.Vertex import Vertex
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color
from topologicpy.Helper import Helper

print("TopologicPy version:", Helper.Version())
renderer = "vscode"


def reset_dictionaries(shell):
    faces = Topology.Faces(shell) or []
    for face in faces:
        dictionary = Topology.Dictionary(face)
        for key in Dictionary.Keys(dictionary) or []:
            if key != "face_id":
                dictionary = Dictionary.RemoveKey(dictionary, key)
        _ = Topology.SetDictionary(face, dictionary)


def transfer_dicts_by_key(topologies, selectors, key):
    topology_lookup = {}
    for topology in topologies or []:
        dictionary = Topology.Dictionary(topology)
        value = Dictionary.ValueAtKey(dictionary, key, None)
        if value is not None:
            topology_lookup[str(value)] = topology

    for selector in selectors or []:
        dictionary = Topology.Dictionary(selector)
        value = Dictionary.ValueAtKey(dictionary, key, None)
        if value is not None and str(value) in topology_lookup:
            _ = Topology.SetDictionary(topology_lookup[str(value)], dictionary)


def vertices_to_frame(vertices, key_fields):
    records = []
    for vertex in vertices or []:
        dictionary = Topology.Dictionary(vertex)
        record = {
            "x": round(Vertex.X(vertex), 3),
            "y": round(Vertex.Y(vertex), 3),
            "z": round(Vertex.Z(vertex), 3),
        }
        for key in key_fields:
            record[key] = Dictionary.ValueAtKey(dictionary, key, None)
        records.append(record)
    return pd.DataFrame(records)

TopologicPy version: The version that you are using (0.9.43) is OLDER than the latest version (0.9.50) from PyPI. Please consider upgrading to the latest version.


In [ ]:
print("TopologicPy version:", Helper.Version())

## Paths and Settings

The notebook reads one of the Homework04 floor plans from the local workspace. If the default file is missing, it falls back to the first available OBJ so the notebook still runs.

In [2]:
BASE = Path(r"C:\Users\etmaglari\IAAC\etmaglari_gML")
FLOORPLAN_DIR = BASE / "Homework04" / "FloorPlans"
OUTPUT_DIR = BASE / "Homework04" / "Notebooks" / "analysis_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AVAILABLE_OBJS = sorted(FLOORPLAN_DIR.glob("*.obj"))
print("Available floor plans:")
for obj_path in AVAILABLE_OBJS:
    print("-", obj_path.name)

OBJ_NAME = "GroungFloor.obj"
OBJ_PATH = FLOORPLAN_DIR / OBJ_NAME
if not OBJ_PATH.exists():
    if not AVAILABLE_OBJS:
        raise FileNotFoundError(f"No OBJ files found in {FLOORPLAN_DIR}")
    OBJ_PATH = AVAILABLE_OBJS[0]

print("Using OBJ:", OBJ_PATH.name)

Available floor plans:
- GroungFloor.obj
- Level1.obj
- Level2.obj
- Level3.obj
Using OBJ: GroungFloor.obj


In [ ]:
print("Floor plan overview:")
for obj_path in AVAILABLE_OBJS:
    floor_objects = Topology.ByOBJPath(str(obj_path))
    print("-", obj_path.name, f"({len(floor_objects)} objects)")
    Topology.Show(
        floor_objects,
        camera=[0, 0, 6],
        faceColor=[210, 210, 250],
        faceOpacity=1,
        edgeColor="white",
        edgeWidth=2,
        showVertices=False,
        backgroundColor="black",
        width=700,
        height=500,
        renderer=renderer,
    )

Floor plan overview:
- GroungFloor.obj (2 objects)


## Graph Construction

This section loads the OBJ, derives the floor face, creates the clipped grid, slices the shell, and builds both the navigation graph and the analysis graph.

## Shortest Path Example

The routing example below uses the navigation graph and mirrors the shortest-path workflow from S03-06. It reports both the network path length and a straightened version of the route.

In [5]:
def pick_extreme_vertices(vertices):
    start_vertex = min(vertices, key=lambda vertex: Vertex.X(vertex) + Vertex.Y(vertex))
    end_vertex = max(vertices, key=lambda vertex: Vertex.X(vertex) + Vertex.Y(vertex))
    return start_vertex, end_vertex


start_vertex, end_vertex = pick_extreme_vertices(Graph.Vertices(navigation_graph) or [])
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
path_start = perf_counter()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
straight_path = Wire.Straighten(shortest_path, host=floor_face)
path_end = perf_counter()

print("Shortest path duration:", round(path_end - path_start, 2), "seconds")
print("Original path length:", round(Wire.Length(shortest_path), 3))
print("Straightened path length:", round(Wire.Length(straight_path), 3))

Topology.Show(
    floor_face,
    shortest_path,
    straight_path,
    camera=[0, 0, 6],
    faceColor=[210, 210, 250],
    faceOpacity=1,
    edgeColor="red",
    edgeWidth=4,
    showVertices=False,
    backgroundColor="black",
    width=900,
    height=650,
    renderer=renderer,
)

Shortest path duration: 0.05 seconds
Original path length: 16.228
Straightened path length: 14.01


## Grid and Shell Construction

This follows the S03-06 / S03-07 pattern: create a clipped grid from the bounding rectangle, slice the floor face, and label each face so graph metrics can be transferred back to the geometry.

In [12]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell) or []
    for face in faces:
        dictionary = Topology.Dictionary(face)
        for key in Dictionary.Keys(dictionary) or []:
            if key != "face_id":
                dictionary = Dictionary.RemoveKey(dictionary, key)
        _ = Topology.SetDictionary(face, dictionary)


def transfer_dicts_by_key(topologies, selectors, key):
    topology_lookup = {}
    for topology in topologies or []:
        dictionary = Topology.Dictionary(topology)
        value = Dictionary.ValueAtKey(dictionary, key, None)
        if value is not None:
            topology_lookup[str(value)] = topology

    for selector in selectors or []:
        dictionary = Topology.Dictionary(selector)
        value = Dictionary.ValueAtKey(dictionary, key, None)
        if value is not None and str(value) in topology_lookup:
            _ = Topology.SetDictionary(topology_lookup[str(value)], dictionary)


def vertices_to_frame(vertices, key_fields):
    records = []
    for vertex in vertices or []:
        dictionary = Topology.Dictionary(vertex)
        record = {
            "x": round(Vertex.X(vertex), 3),
            "y": round(Vertex.Y(vertex), 3),
            "z": round(Vertex.Z(vertex), 3),
        }
        for key in key_fields:
            record[key] = Dictionary.ValueAtKey(dictionary, key, None)
        records.append(record)
    return pd.DataFrame(records)

graph_vertices = Graph.Vertices(analysis_graph) or []
degree_values = Graph.DegreeCentrality(analysis_graph, normalize=True)
closeness_values = Graph.ClosenessCentrality(analysis_graph)
betweenness_values = Graph.BetweennessCentrality(analysis_graph, normalize=True)

degree_min = min(degree_values)
degree_max = max(degree_values)
closeness_min = min(closeness_values)
closeness_max = max(closeness_values)
betweenness_min = min(betweenness_values)
betweenness_max = max(betweenness_values)

for vertex, value in zip(graph_vertices, degree_values):
    dictionary = Topology.Dictionary(vertex)
    color = Color.AnyToHex(Color.ByValueInRange(value, minValue=degree_min, maxValue=degree_max, colorScale="thermal"))
    dictionary = Dictionary.SetValueAtKey(dictionary, "degree_centrality", value)
    dictionary = Dictionary.SetValueAtKey(dictionary, "dc_color", color)
    _ = Topology.SetDictionary(vertex, dictionary)
degree_frame = vertices_to_frame(graph_vertices, ["face_id", "degree_centrality", "dc_color"])
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, graph_vertices, "face_id")
print("Top degree-centrality faces:")
display(degree_frame.sort_values("degree_centrality", ascending=False).head(10))
Topology.Show(
    faces,
    faceColorKey="dc_color",
    faceOpacity=1,
    showEdges=False,
    showVertices=False,
    camera=[0, 0, 6],
    backgroundColor="black",
    width=900,
    height=650,
    renderer=renderer,
)

for vertex, value in zip(graph_vertices, closeness_values):
    dictionary = Topology.Dictionary(vertex)
    color = Color.AnyToHex(Color.ByValueInRange(value, minValue=closeness_min, maxValue=closeness_max, colorScale="thermal"))
    dictionary = Dictionary.SetValueAtKey(dictionary, "closeness_centrality", value)
    dictionary = Dictionary.SetValueAtKey(dictionary, "cc_color", color)
    _ = Topology.SetDictionary(vertex, dictionary)
closeness_frame = vertices_to_frame(graph_vertices, ["face_id", "closeness_centrality", "cc_color"])
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, graph_vertices, "face_id")
print("Top closeness-centrality faces:")
display(closeness_frame.sort_values("closeness_centrality", ascending=False).head(10))
Topology.Show(
    faces,
    faceColorKey="cc_color",
    faceOpacity=1,
    showEdges=False,
    showVertices=False,
    camera=[0, 0, 6],
    backgroundColor="black",
    width=900,
    height=650,
    renderer=renderer,
)

for vertex, value in zip(graph_vertices, betweenness_values):
    dictionary = Topology.Dictionary(vertex)
    color = Color.AnyToHex(Color.ByValueInRange(value, minValue=betweenness_min, maxValue=betweenness_max, colorScale="thermal"))
    dictionary = Dictionary.SetValueAtKey(dictionary, "betweenness_centrality", value)
    dictionary = Dictionary.SetValueAtKey(dictionary, "bc_color", color)
    _ = Topology.SetDictionary(vertex, dictionary)
betweenness_frame = vertices_to_frame(graph_vertices, ["face_id", "betweenness_centrality", "bc_color"])
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, graph_vertices, "face_id")
print("Top betweenness-centrality faces:")
display(betweenness_frame.sort_values("betweenness_centrality", ascending=False).head(10))
Topology.Show(
    faces,
    faceColorKey="bc_color",
    faceOpacity=1,
    showEdges=False,
    showVertices=False,
    camera=[0, 0, 6],
    backgroundColor="black",
    width=900,
    height=650,
    renderer=renderer,
)

Top degree-centrality faces:


,x,y,z,face_id,degree_centrality,dc_color
6,-0.847,11.914,-0.25,face_7,0.285714,#E7FA5A
10,-1.695,8.779,-0.25,face_11,0.285714,#E7FA5A
2,1.309,17.790,-0.25,face_3,0.214286,#E87957
5,1.317,11.739,-0.25,face_6,0.214286,#E87957
4,1.402,14.845,-0.25,face_5,0.214286,#E87957
7,1.309,8.790,-0.25,face_8,0.214286,#E87957
1,-1.678,17.790,-0.25,face_2,0.214286,#E87957
3,-1.678,15.531,-0.25,face_4,0.142857,#724792
0,-0.178,19.342,-0.25,face_1,0.142857,#724792
8,-0.063,13.205,-0.25,face_9,0.142857,#724792


Top closeness-centrality faces:


,x,y,z,face_id,closeness_centrality,cc_color
5,1.317,11.739,-0.25,face_6,0.466667,#E7FA5A
6,-0.847,11.914,-0.25,face_7,0.451613,#F2DB4A
4,1.402,14.845,-0.25,face_5,0.400000,#EC7F51
7,1.309,8.790,-0.25,face_8,0.400000,#EC7F51
10,-1.695,8.779,-0.25,face_11,0.388889,#DF7061
9,-1.678,13.955,-0.25,face_10,0.333333,#804E8E
8,-0.063,13.205,-0.25,face_9,0.333333,#804E8E
2,1.309,17.790,-0.25,face_3,0.318182,#664295
3,-1.678,15.531,-0.25,face_4,0.311111,#5A3D99
11,1.309,5.790,-0.25,face_12,0.304348,#4C3899


Top betweenness-centrality faces:


,x,y,z,face_id,betweenness_centrality,bc_color
5,1.317,11.739,-0.25,face_6,0.531136,#E7FA5A
6,-0.847,11.914,-0.25,face_7,0.456044,#F8BD40
4,1.402,14.845,-0.25,face_5,0.445055,#F9B53E
10,-1.695,8.779,-0.25,face_11,0.287546,#BF6379
7,1.309,8.790,-0.25,face_8,0.216117,#8D528B
2,1.309,17.790,-0.25,face_3,0.181319,#754991
8,-0.063,13.205,-0.25,face_9,0.065934,#1B3078
9,-1.678,13.955,-0.25,face_10,0.065934,#1B3078
3,-1.678,15.531,-0.25,face_4,0.060440,#173071
11,1.309,5.790,-0.25,face_12,0.027473,#082A4E


## Metrics Export and Interpretation

Use the table below to write the presentation narrative. The notebook exports one CSV with the most important spatial-graph measures so the results can be reused outside VS Code.

## Metrics Export and Interpretation

Use the table below to write the presentation narrative. The notebook exports one CSV with the most important spatial-graph measures so the results can be reused outside VS Code.

1. Accessibility: which spaces have the highest closeness centrality?
2. Circulation: which spaces have the highest betweenness centrality?
3. Local hubs: where is degree centrality concentrated?
4. Shortest route: what does the chosen path tell you about spatial continuity?
5. Segregation: which parts of the plan stay weakly connected?

## Presentation Notes

Suggested closing points for the final submission:
- mention the floor plan used and the chosen grid step,
- identify the top faces by closeness, betweenness, and degree,
- describe one shortest-path observation,
- and note one limitation of the graph construction that could be improved in a next iteration.

## Utility Functions

These helpers follow the same TopologicPy patterns used in the class notebooks: dictionary cleanup, dictionary transfer by shared keys, and a small face-table exporter for reporting.

In [ ]:
def load_floor_face(obj_path):
    objects = Topology.ByOBJPath(str(obj_path))
    if not objects:
        raise RuntimeError(f"Could not import OBJ file: {obj_path}")

    floor_face = None
    for obj in objects:
        faces = Topology.Faces(obj) or []
        if faces:
            floor_face = faces[0]
            break
        wires = Topology.Wires(obj) or []
        if wires:
            floor_face = Face.ByWire(wires[0])
            if floor_face:
                break

    if floor_face is None:
        raise RuntimeError("Could not derive a floor face from the imported OBJ geometry.")

    return objects, floor_face


objects, floor_face = load_floor_face(OBJ_PATH)
print(f"Imported objects: {len(objects)}")

bounding_rectangle = Wire.BoundingRectangle(floor_face)
bounds = Topology.Dictionary(bounding_rectangle)
xmin = Dictionary.ValueAtKey(bounds, "xmin")
xmax = Dictionary.ValueAtKey(bounds, "xmax")
ymin = Dictionary.ValueAtKey(bounds, "ymin")
ymax = Dictionary.ValueAtKey(bounds, "ymax")
width = Dictionary.ValueAtKey(bounds, "width")
length = Dictionary.ValueAtKey(bounds, "length")

print("Bounds:")
print("  xmin:", round(xmin, 3), "xmax:", round(xmax, 3))
print("  ymin:", round(ymin, 3), "ymax:", round(ymax, 3))
print("  width:", round(width, 3), "length:", round(length, 3))

Topology.Show(
    floor_face,
    camera=[0, 0, 6],
    faceColor=[210, 210, 250],
    faceOpacity=1,
    edgeColor="white",
    edgeWidth=3,
    showVertices=False,
    backgroundColor="black",
    width=900,
    height=650,
    renderer=renderer,
)

GRID_STEP = 1
uRange = list(range(0, int(width) + GRID_STEP, GRID_STEP))
vRange = list(range(0, int(length) + GRID_STEP, GRID_STEP))

grid = Grid.EdgesByDistances(floor_face, clip=True, uRange=uRange, vRange=vRange)
shell = Topology.Slice(floor_face, grid)
faces = Topology.Faces(shell) or []

for index, face in enumerate(faces):
    dictionary = Dictionary.ByKeyValue("face_id", f"face_{index + 1}")
    _ = Topology.SetDictionary(face, dictionary)

navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)
graph_vertices = Graph.Vertices(analysis_graph) or []

print("Grid step:", GRID_STEP)
print("Sliced faces:", len(faces))
print("Navigation graph vertices:", len(Graph.Vertices(navigation_graph) or []))
print("Analysis graph vertices:", len(graph_vertices))

Topology.Show(
    floor_face,
    grid,
    camera=[0, 0, 6],
    faceColor=[210, 210, 250],
    faceOpacity=0.85,
    edgeColor="grey",
    edgeWidth=2,
    showVertices=False,
    backgroundColor="black",
    width=900,
    height=650,
    renderer=renderer,
)

Topology.Show(
    shell,
    camera=[0, 0, 6],
    faceColor=[230, 230, 230],
    faceOpacity=1,
    edgeColor="grey",
    edgeWidth=1,
    showVertices=False,
    backgroundColor="black",
    width=900,
    height=650,
    renderer=renderer,
)

## Visibility Graph Analysis

This follows the S03-08 pattern with isovists sampled on a denser grid of viewpoints. The result is a visibility score for each viewpoint, colored from low to high visibility.

In [ ]:
VISIBILITY_STEP = 2
vis_uRange = list(range(0, int(width) + VISIBILITY_STEP, VISIBILITY_STEP))
vis_vRange = list(range(0, int(length) + VISIBILITY_STEP, VISIBILITY_STEP))
visibility_grid = Grid.VerticesByDistances(floor_face, clip=True, uRange=vis_uRange, vRange=vis_vRange)
visibility_vertices = Topology.Vertices(visibility_grid) or []
visibility_samples = []

for vertex in visibility_vertices:
    isovist = Face.Isovist(floor_face, vertex)
    if isovist:
        visible_vertices = Vertex.IsInternal2D(graph_vertices, isovist) or []
        visible_vertices = [v for v in visible_vertices if v]
        visibility_samples.append((vertex, isovist, len(visible_vertices)))

if visibility_samples:
    min_visibility = min(sample[2] for sample in visibility_samples)
    max_visibility = max(sample[2] for sample in visibility_samples)
    isovists = []
    for vertex, isovist, visibility_count in visibility_samples:
        dictionary = Dictionary.ByKeyValue("visibility", visibility_count)
        color = Color.AnyToHex(Color.ByValueInRange(visibility_count, minValue=min_visibility, maxValue=max_visibility, colorScale="thermal"))
        dictionary = Dictionary.SetValueAtKey(dictionary, "vb_color", color)
        dictionary = Dictionary.SetValueAtKey(dictionary, "size", 14)
        _ = Topology.SetDictionary(vertex, dictionary)
        isovists.append(isovist)

    visibility_frame = vertices_to_frame(visibility_vertices, ["visibility", "vb_color"])
    print("Visibility samples:", len(visibility_samples))
    display(visibility_frame.sort_values("visibility", ascending=False).head(10))
    Topology.Show(
        floor_face,
        isovists,
        camera=[0, 0, 6],
        faceOpacity=0.35,
        showEdges=False,
        showVertices=False,
        backgroundColor="black",
        width=900,
        height=650,
        renderer=renderer,
    )
    Topology.Show(
        floor_face,
        visibility_vertices,
        camera=[0, 0, 6],
        faceOpacity=1,
        showEdges=False,
        showVertices=True,
        vertexColorKey="vb_color",
        vertexSizeKey="size",
        backgroundColor="black",
        width=900,
        height=650,
        renderer=renderer,
    )
else:
    print("No visibility samples were created.")